Cell 0 – Install

In [ ]:
!pip install torch matplotlib --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.6 MB/s eta 0:00:00


Cell 1 – Imports

In [ ]:
# Core libraries
import math
import random
import argparse
import collections
from copy import deepcopy
from pathlib import Path                # handy file I/O

# Scientific computing
import numpy as np
import matplotlib.pyplot as plt

# PyTorch — used for the value‑function network
import torch
import torch.nn as nn
import torch.optim as optim

# Optional: make sure all tensors default to float32 on CPU,
# which avoids type‑mismatch bugs in TD targets.
torch.set_default_dtype(torch.float32)

# Seed helper for full reproducibility (optional; comment out
# if you prefer different runs every time).
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# Make matplotlib a bit prettier in notebooks
plt.style.use("ggplot")


Cell 2 – Globals (Environment settings & RL constants)

In [ ]:
# Grid‑world geometry
GRID_W, GRID_H   = 15, 10           # width × height
NUM_MOUNTAINS    = 10               # impassable obstacles
MAX_TURNS        = 30               # episode horizon

# Aircraft weapon system
MAX_MISSILES     = 4                # hard‑cap capacity
INIT_MISSILES    = 3                # starts with 3 (was 2 in HW‑1)

# Discount factor
GAMMA            = 0.93             # slightly larger → rewards propagate further

#  Reward design
# Killing a drone is now more valuable to encourage aggressive play.
R_GOAL  = 1000                      # eliminate all drones + reach goal
R_KILL  = 350                       # per‑drone kill bonus   (was 250)
R_LOSS  = -1000                     # aircraft captured / surrounded


Cell 3 – Environment (DroneEnv)

In [ ]:
# Grid‑world simulator identical to HW‑1 *except*:
# • Aircraft starts with 3 missiles (INIT_MISSILES)
# • Reload‑zone replenishes +3 missiles **only after ≥3 shots**,
#   then respawns elsewhere.
# • Each drone moves greedily but independently toward the aircraft.


DIRS = [(0, -1),   # 0: UP
        (0,  1),   # 1: DOWN
        (-1, 0),   # 2: LEFT
        (1,  0)]   # 3: RIGHT


def _clamp(v: int, lo: int, hi: int) -> int:
    """Keep v inside [lo, hi]."""
    return max(lo, min(v, hi))


class DroneEnv:
    """15×10 grid, 1 aircraft, n suicide‑drones, 10 mountains, 1 reload‑zone."""
    def __init__(self, n_drones: int = 4, seed: int | None = None):
        self.n_d = n_drones
        self.rng = random.Random(seed)
        self.reset()

    # helpers
    def _rand_empty(self) -> tuple[int, int]:
        """Return a random empty cell: not mountain / goal / occupied."""
        while True:
            p = (self.rng.randint(0, GRID_W - 1),
                 self.rng.randint(0, GRID_H - 1))
            if (p in self.mountains) or (p == self.goal):
                continue
            if p == getattr(self, "air", None) or p in getattr(self, "drones", []):
                continue
            return p

    # public API
    def reset(self) -> dict:
        """Start a new episode; return initial state‑dict."""
        self.turn = 0
        self.goal = (GRID_W - 1, GRID_H - 1)

        # Mountains
        self.mountains: set[tuple[int, int]] = set()
        while len(self.mountains) < NUM_MOUNTAINS:
            p = (self.rng.randint(0, GRID_W - 1),
                 self.rng.randint(0, GRID_H - 1))
            if p != self.goal:
                self.mountains.add(p)

        # Entities
        self.reload = self._rand_empty()
        self.air    = self._rand_empty()
        self.miss   = INIT_MISSILES
        self.drones = [self._rand_empty() for _ in range(self.n_d)]

        return self.state()

    def state(self) -> dict:
        """Return a (shallow‑copy) dict describing current state."""
        return {
            "air":    self.air,
            "miss":   self.miss,
            "drones": tuple(self.drones),
            "turn":   self.turn,
            "reload": self.reload,
        }

    # internal movement
    def _move(self, pos: tuple[int, int], dir_idx: int) -> tuple[int, int]:
        dx, dy = DIRS[dir_idx]
        return _clamp(pos[0] + dx, 0, GRID_W - 1), \
               _clamp(pos[1] + dy, 0, GRID_H - 1)

    def _drone_dir(self, dpos: tuple[int, int]) -> int:
        """Greedy direction (Chebyshev) toward aircraft."""
        ax, ay = self.air
        dx, dy = dpos
        if abs(ax - dx) > abs(ay - dy):
            return 3 if ax > dx else 2      # → or ←
        return 1 if ay > dy else 0          # ↓ or ↑

    # environment step
    def step(self, action: int) -> tuple[dict, float, bool]:
        """
        action ∈ {0‑3 moves, 4+idx shoot}.  Moves first, then all drones move.
        Returns (new_state, reward, done).
        """
        reward = 0.0

        # aircraft
        if action < 4:                      # movement
            self.air = self._move(self.air, action)
        else:                               # shooting
            if self.miss > 0:
                idx = action - 4
                if idx < len(self.drones):
                    # LOS simplified to Chebyshev distance ≤ 2
                    if max(abs(self.drones[idx][0] - self.air[0]),
                           abs(self.drones[idx][1] - self.air[1])) <= 2:
                        self.drones.pop(idx)
                        reward += R_KILL
                self.miss -= 1               # missile spent

        # reload zone
        if self.air == self.reload and self.miss <= MAX_MISSILES - 3:
            # Recharge THREE missiles and relocate zone
            self.miss = min(MAX_MISSILES, self.miss + 3)
            self.reload = self._rand_empty()

        # drones move (greedy, independent)
        for i in range(len(self.drones)):
            self.drones[i] = self._move(self.drones[i],
                                         self._drone_dir(self.drones[i]))

        # bookkeeping
        self.turn += 1
        done = self._terminal()
        if done:
            reward += R_GOAL if (not self.drones and self.air == self.goal) else R_LOSS
        return self.state(), reward, done

    # termination
    def _terminal(self) -> bool:
        if self.turn >= MAX_TURNS:                             # time‑out
            return True
        if self.air in self.drones:                            # captured
            return True
        if (not self.drones) and (self.air == self.goal):      # victory
            return True
        return False


Cell 4 – Feature Extractor φ

In [ ]:
def phi(s: dict) -> np.ndarray:
    ax, ay = s["air"]

    # global, scalar features
    goal_dist   = math.hypot(GRID_W - 1 - ax, GRID_H - 1 - ay) \
                  / math.hypot(GRID_W, GRID_H)          # ∈ [0,1]
    missiles    = s["miss"] / MAX_MISSILES              # ∈ [0,1]
    drone_ratio = len(s["drones"]) / 4                  # 0,0.25,…,1
    turn_frac   = s["turn"] / MAX_TURNS                 # progress bar

    base = [goal_dist, missiles, drone_ratio, turn_frac]

    # per‑drone radial distances (padded with 1.0)
    dists = [1.0] * 4
    for i, (dx, dy) in enumerate(s["drones"]):
        dists[i] = math.hypot(ax - dx, ay - dy) / math.hypot(GRID_W, GRID_H)

    nearest = min(dists)

    # drones within Chebyshev‑range ≤2 (shootable)
    in_range = sum(
        1
        for dx, dy in s["drones"]
        if max(abs(ax - dx), abs(ay - dy)) <= 2
    ) / 4.0                                             # normalize

    # mean heading of swarm (helps with surround detection)
    if s["drones"]:
        avg_dx = np.mean([dx - ax for dx, _ in s["drones"]]) / GRID_W
        avg_dy = np.mean([dy - ay for _, dy in s["drones"]]) / GRID_H
    else:
        avg_dx = avg_dy = 0.0

    feature_vec = np.array(
        base + dists + [nearest, in_range, avg_dx, avg_dy],
        dtype=np.float32,
    )
    return feature_vec


In [ ]:
class LinV:
    def __init__(self, dim: int):
        self.w = np.zeros(dim, np.float32)
    def __call__(self, f: np.ndarray) -> float:
        return float(self.w.dot(f))
    def update(self, f, target, alpha: float):
        self.w += alpha * (target - self(f)) * f

def train_baseline(n_drones: int, episodes: int = 6000, alpha: float = 0.02):
    """Semi‑gradient TD‑0 with random policy (baseline)."""
    env = DroneEnv(n_drones)
    V   = LinV(len(phi(env.state())))
    curve = []

    for ep in range(episodes):
        s = env.reset(); done = False; G = 0.0
        while not done:
            a = random.choice(range(4))           # pure random
            s2, r, done = env.step(a)
            target = r if done else r + GAMMA * V(phi(s2))
            V.update(phi(s), target, alpha)
            G += r; s = s2
        curve.append(G)
    return V, curve


Cell 5 : Non‑Linear V(s) + ε-greedy PI

In [ ]:
class Vnet(nn.Module):
    """2‑layer MLP:  input‑dim → 128 → 64 → 1  (ReLU)"""
    def __init__(self, dim: int):
        super().__init__()
        self.m = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.m(x.float()).squeeze(-1)


# helper
to_f = lambda arr: torch.tensor(arr, dtype=torch.float32, device="cpu")


# valid actions
def actions(env):
    """Return list of valid aircraft actions in current state."""
    acts = list(range(4))                           # moves
    if env.miss > 0:                                # shoot only if missile
        acts.extend(
            4 + i
            for i, d in enumerate(env.drones)
            if max(abs(env.air[0] - d[0]), abs(env.air[1] - d[1])) <= 2
        )
    return acts or [0]                              # fallback: move‑UP


# greedy
def greedy(env, val_fn):
    best_a, best_v = 0, -1e9
    for a in actions(env):
        e = deepcopy(env)
        s2, r, done = e.step(a)
        v = r if done else r + GAMMA * val_fn(phi(s2))
        if v > best_v:
            best_v, best_a = v, a
    return best_a


# training
def train_PI(
    n_drones: int,
    episodes: int = 20000,
    lr: float = 3e-4,
    net_prev: Vnet | None = None,
    batch: int = 256,
    capacity: int = 20_000,
):
    """
    Replay‑buffer TD(0) + ε‑greedy improvement.
    Returns (trained_net, list_of_returns).
    """
    env = DroneEnv(n_drones)
    net = net_prev if net_prev else Vnet(len(phi(env.state())))
    opt = optim.Adam(net.parameters(), lr=lr)

    # Experience replay buffer  (s, a, r, s', done)
    memory = collections.deque(maxlen=capacity)
    returns = []

    for ep in range(episodes):
        s = env.reset()
        done, G = False, 0.0

        ε = 0.05 + 0.25 * math.exp(-ep / 9000)      # slower decay

        while not done:
            # choose action (ε‑greedy)
            if random.random() < ε:
                a = random.choice(actions(env))
            else:
                a = greedy(env, lambda f: net(to_f(f)).item())

            s2, r, done = env.step(a)
            memory.append((phi(s), a, r, phi(s2), done))

            # replay learning
            if len(memory) >= batch:
                B   = random.sample(memory, batch)

                ss  = torch.stack([to_f(b[0]) for b in B])          # [B, dim]
                rr  = torch.tensor([b[2] for b in B], dtype=torch.float32)
                ss2 = torch.stack([to_f(b[3]) for b in B])
                dd  = torch.tensor([b[4] for b in B], dtype=torch.float32)

                v_s  = net(ss).squeeze()            # ← shape [B]
                with torch.no_grad():
                    v_s2 = net(ss2).squeeze()       # bootstrap
                    target = rr + GAMMA * (1.0 - dd) * v_s2

                loss = nn.functional.mse_loss(v_s, target)
                opt.zero_grad(); loss.backward(); opt.step()

            G += r
            s = s2

        returns.append(G)

    return net, returns


Cell 6 – Win‑Rate

In [ ]:
@torch.no_grad()                       # disable grad for entire function
def avg_return(val_fn, n_drones: int, episodes: int = 1000) -> float:
    """
    Average *cumulative* reward per episode over `episodes` eval runs.
    Useful for comparing learning curves.
    """
    env = DroneEnv(n_drones)
    total = 0.0

    for _ in range(episodes):
        s, done, G = env.reset(), False, 0.0
        while not done:
            a = greedy(env, val_fn)
            s, r, done = env.step(a)
            G += r
        total += G

    return total / episodes


@torch.no_grad()
def win_rate(val_fn, n_drones: int, episodes: int = 600) -> float:
    """
    Fraction of episodes in which aircraft destroys ALL drones
    *and* reaches the goal (strict victory condition).
    """
    env = DroneEnv(n_drones)
    wins = 0

    for _ in range(episodes):
        env.reset()
        done = False
        while not done:
            a = greedy(env, val_fn)
            _, _, done = env.step(a)
        if env.air == env.goal and not env.drones:
            wins += 1

    return wins / episodes


Cell 7 – Curriculum Training

In [ ]:
def curriculum(
    max_drones: int = 4,
    epi_easy: int = 12_000,
    epi_hard: int = 25_000,
    lr_easy: float = 5e-4,
    lr_hard: float = 3e-4,
) -> dict:
    """
    Returns a dict with keys:
        'nets'   : list[Vnet]   (one per stage, last = final)
        'curves' : list[list]   training returns for each stage
        'avgR'   : list[float]  average cumulative reward @eval
        'wins'   : list[float]  win‑rate fraction
    """
    net, curves, avgR, wins = None, [], [], []

    for nd in range(1, max_drones + 1):
        print(f"\n[Stage {nd}/{max_drones}]  training with {nd} drone(s)")
        episodes = epi_easy if nd <= 2 else epi_hard
        lr       = lr_easy  if nd <= 2 else lr_hard

        net, c = train_PI(nd, episodes=episodes, lr=lr, net_prev=net)
        curves.append(c)

        # Wrap net as val_fn for evaluation
        val_fn = lambda f, N=net: N(to_f(f)).item()

        r_bar   = avg_return(val_fn, nd, episodes=800)
        w_rate  = win_rate(val_fn, nd, episodes=800)

        avgR.append(r_bar)
        wins.append(w_rate)

        print(f"   Avg‑Return: {r_bar:7.1f}   |   Win‑Rate: {w_rate*100:5.1f}%")

    return {"nets": [net], "curves": curves, "avgR": avgR, "wins": wins}


# optional: quick plot helper inside curriculum
def _plot_learning(curves, max_last=300):
    plt.figure(figsize=(6,4))
    for nd, c in enumerate(curves, 1):
        plt.plot(c[-max_last:], label=f"{nd} drone(s)")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.title("Policy‑Iteration – last training window")
    plt.legend(); plt.tight_layout()
    Path("plots").mkdir(exist_ok=True)
    plt.savefig("plots/learning_curves.png", dpi=120)
    plt.close()


Cell 8 – Stage‑Wise Training + Plot

In [ ]:
def stagewise(
    max_drones: int = 4,
    epi_base: int = 6_000,           # for baseline
    epi_easy: int = 12_000, epi_hard: int = 25_000,
):
    # storage
    base_ret, pi_ret   = [], []
    base_wins, pi_wins = [], []
    curves_b, curves_pi = [], []

    net = None   # PI network carried stage → stage

    for nd in range(1, max_drones + 1):
        print(f"\n=== Stage {nd} drone(s) ===")

        # Baseline (TD‑Linear)
        print("  [Baseline] training …")
        Vb, cb = train_baseline(nd, episodes=epi_base)
        curves_b.append(cb)

        val_fn_b = lambda f, V=Vb: V(f)
        r_b  = avg_return(val_fn_b, nd, episodes=600)
        w_b  = win_rate(val_fn_b, nd, episodes=600)
        base_ret.append(r_b); base_wins.append(w_b)

        # Policy‑Iteration (MLP‑PI)
        print("  [PI] training …")
        episodes = epi_easy if nd <= 2 else epi_hard
        lr       = 5e-4 if nd <= 2 else 3e-4
        net, cp  = train_PI(nd, episodes=episodes, lr=lr, net_prev=net)
        curves_pi.append(cp)

        val_fn_pi = lambda f, N=net: N(to_f(f)).item()
        r_pi = avg_return(val_fn_pi, nd, episodes=600)
        w_pi = win_rate(val_fn_pi, nd, episodes=600)
        pi_ret.append(r_pi); pi_wins.append(w_pi)

        print(f"      Avg‑Return  Baseline: {r_b:7.1f} | PI: {r_pi:7.1f}")
        print(f"      Win‑Rate    Baseline: {w_b*100:6.1f}% | PI: {w_pi*100:6.1f}%")

    # Plot learning curves (last 300)
    plt.figure(figsize=(6,4))
    for nd, (cB, cP) in enumerate(zip(curves_b, curves_pi),1):
        plt.plot(cB[-300:], label=f'Base {nd}d', alpha=.4, lw=1)
        plt.plot(cP[-300:], label=f'PI {nd}d',  lw=1.5)
    plt.xlabel("Episode"); plt.ylabel("Return")
    plt.title("Learning curves – last 300 episodes")
    plt.legend(); plt.tight_layout()
    Path("plots").mkdir(exist_ok=True)
    plt.savefig("plots/learning_curves.png", dpi=120)
    plt.close()

    # Plot Win‑Rate
    x = range(1, max_drones+1)
    plt.figure(figsize=(4,3))
    plt.plot(x, base_wins, '-o', label='Baseline')
    plt.plot(x, pi_wins,   '-o', label='Policy Iteration')
    plt.xticks(x); plt.ylim(0,1)
    plt.xlabel("# Drones"); plt.ylabel("Win‑Rate")
    plt.title("Win‑Rate vs Drone Count")
    plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
    plt.savefig("plots/winrate_vs_drones.png", dpi=120)
    plt.close()

    return {
        "baseline": {"ret": base_ret, "win": base_wins},
        "pi":       {"ret": pi_ret,   "win": pi_wins},
    }


Cell 9 – Main

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--fast", action="store_true",
                        help="quick run (tiny #episodes) just to verify code")
    args, _ = parser.parse_known_args()

    # choose episode budget based on --fast
    if args.fast:
        print("⚡ FAST mode – tiny training budget (results not meaningful)")
        result = stagewise(max_drones=4, epi_base=1500,
                           epi_easy=3000, epi_hard=6000)
    else:
        result = stagewise()          # defaults: 6k baseline, 12k / 25k PI

    # pretty print summary
    print("\n=== Average cumulative reward over 1 k eval episodes ===")
    print("   drones | Baseline | Policy‑Iter")
    print(" -------- | -------- | -----------")
    for nd, (b, p) in enumerate(
        zip(result["baseline"]["ret"], result["pi"]["ret"]), 1
    ):
        print(f"     {nd:<5} | {b:8.1f} | {p:11.1f}")

    print("\n=== Win‑Rate (strict victory) ===")
    print("   drones | Baseline | Policy‑Iter")
    print(" -------- | -------- | -----------")
    for nd, (b, p) in enumerate(
        zip(result["baseline"]["win"], result["pi"]["win"]), 1
    ):
        print(f"     {nd:<5} | {b*100:7.1f}% | {p*100:10.1f}%")

    print("\nPlots saved in ./plots/")



=== Stage 1 drone(s) ===
  [Baseline] training …
  [PI] training …
      Avg‑Return  Baseline:   920.0 | PI:  1350.0
      Win‑Rate    Baseline:   81.3% | PI:  100.0%

=== Stage 2 drone(s) ===
  [Baseline] training …
  [PI] training …
      Avg‑Return  Baseline:  1193.3 | PI:  1700.0
      Win‑Rate    Baseline:   80.7% | PI:  100.0%

=== Stage 3 drone(s) ===
  [Baseline] training …
  [PI] training …
      Avg‑Return  Baseline:  1498.2 | PI:  2045.5
      Win‑Rate    Baseline:   68.5% | PI:   99.7%

=== Stage 4 drone(s) ===
  [Baseline] training …
  [PI] training …
      Avg‑Return  Baseline:    49.8 | PI:   602.8
      Win‑Rate    Baseline:    0.5% | PI:   21.7%

=== Average cumulative reward over 1 k eval episodes ===
   drones | Baseline | Policy‑Iter
 -------- | -------- | -----------
     1     |    920.0 |      1350.0
     2     |   1193.3 |      1700.0
     3     |   1498.2 |      2045.5
     4     |     49.8 |       602.8

=== Win‑Rate (strict victory) ===
   drones | Baseline 